In [1]:
print("Check_Python")
%load_ext autotime

Check_Python
time: 15 ms (started: 2026-04-17 17:29:50 +05:30)


In [2]:
import os
from pathlib import Path
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

time: 10 s (started: 2026-04-17 17:29:50 +05:30)


In [3]:
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

print(TRAIN_DIR.exists(), VAL_DIR.exists(), TEST_DIR.exists())

True True True
time: 0 ns (started: 2026-04-17 17:30:00 +05:30)


In [4]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
COLOR_MODE = "grayscale"

time: 0 ns (started: 2026-04-17 17:30:00 +05:30)


In [5]:
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary",
    shuffle=False 
)

Found 624 images belonging to 2 classes.
time: 78 ms (started: 2026-04-17 17:30:00 +05:30)


In [6]:
test_generator.reset()
y_true = test_generator.classes

time: 16 ms (started: 2026-04-17 17:30:00 +05:30)


## Evalute Class_weight

In [7]:
model_class_weights = load_model("best_model_class_weights.h5")
result_class_weights = model_class_weights.evaluate(test_generator, verbose=1)

20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 274ms/step - accuracy: 0.7324 - loss: 1.0950
time: 7.97 s (started: 2026-04-17 17:30:00 +05:30)


In [8]:
raw_preds_class_weights = model_class_weights.predict(test_generator).flatten()

best_thresh_class_weights, best_f1_class_weights = 0.5, 0

for t in np.arange(0.2, 0.8, 0.01):
    preds_class_weights = (raw_preds_class_weights > t).astype(int)
    f1_class_weights = f1_score(y_true, preds_class_weights)
    
    if f1_class_weights > best_f1_class_weights:
        best_f1_class_weights = f1_class_weights
        best_thresh_class_weights = t

print(f"Best threshold: {best_thresh_class_weights:.2f}, F1: {best_f1_class_weights:.4f}")

20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 252ms/step
Best threshold: 0.79, F1: 0.8400
time: 6.58 s (started: 2026-04-17 17:30:08 +05:30)


In [9]:
y_pred_class_weights = (model_class_weights.predict(test_generator) > best_thresh_class_weights).astype(int)

20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 252ms/step
time: 6.23 s (started: 2026-04-17 17:30:15 +05:30)


In [10]:
print("Accuracy0:", (result_class_weights[1]))

Accuracy0: 0.7323718070983887
time: 0 ns (started: 2026-04-17 17:30:21 +05:30)


In [11]:
print("Report0:\n", classification_report(y_true, y_pred_class_weights))

Report0:
               precision    recall  f1-score   support

           0       0.85      0.48      0.62       234
           1       0.75      0.95      0.84       390

    accuracy                           0.77       624
   macro avg       0.80      0.72      0.73       624
weighted avg       0.79      0.77      0.76       624

time: 16 ms (started: 2026-04-17 17:30:21 +05:30)


In [12]:
print("Unique predictions:", np.unique(y_pred_class_weights, return_counts=True))
print("True distribution: ", np.unique(y_true, return_counts=True))

Unique predictions: (array([0, 1]), array([133, 491]))
True distribution:  (array([0, 1], dtype=int32), array([234, 390]))
time: 0 ns (started: 2026-04-17 17:30:21 +05:30)


In [13]:
print("Report:\n",confusion_matrix(y_true, y_pred_class_weights))

Report:
 [[113 121]
 [ 20 370]]
time: 0 ns (started: 2026-04-17 17:30:21 +05:30)


## Evalute OverSampling

In [14]:
test_generator.reset()
y_true = test_generator.classes

time: 0 ns (started: 2026-04-17 17:30:21 +05:30)


In [15]:
model_OverSampling = load_model("best_model_OverSampling.h5")
result_OverSampling = model_OverSampling.evaluate(test_generator, verbose=1)

20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 248ms/step - accuracy: 0.7853 - loss: 0.6439
time: 7.17 s (started: 2026-04-17 17:30:21 +05:30)


In [16]:
test_generator.reset()
raw_preds_OverSampling = model_OverSampling.predict(test_generator).flatten()

best_thresh_OverSampling, best_f1_OverSampling = 0.5, 0

for t in np.arange(0.2, 0.8, 0.01):
    preds_OverSampling = (raw_preds_OverSampling > t).astype(int)
    f1_OverSampling = f1_score(y_true, preds_OverSampling)
    
    if f1_OverSampling > best_f1_OverSampling:
        best_f1_OverSampling = f1_OverSampling
        best_thresh_OverSampling = t

print(f"Best threshold: {best_thresh_OverSampling:.2f}, F1: {best_f1_OverSampling:.4f}")

20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 259ms/step
Best threshold: 0.66, F1: 0.8681
time: 6.67 s (started: 2026-04-17 17:30:28 +05:30)


In [17]:
y_pred_OverSampling = (model_OverSampling.predict(test_generator) > best_thresh_OverSampling).astype(int)

20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 253ms/step
time: 6.24 s (started: 2026-04-17 17:30:35 +05:30)


In [18]:
print("Accuracy0:", (result_OverSampling[1]))

Accuracy0: 0.7852563858032227
time: 0 ns (started: 2026-04-17 17:30:41 +05:30)


In [19]:
print("Report0:\n", classification_report(y_true, y_pred_OverSampling))

Report0:
               precision    recall  f1-score   support

           0       0.90      0.58      0.70       234
           1       0.79      0.96      0.87       390

    accuracy                           0.82       624
   macro avg       0.85      0.77      0.79       624
weighted avg       0.83      0.82      0.81       624

time: 15 ms (started: 2026-04-17 17:30:41 +05:30)


In [20]:
print("Unique predictions:", np.unique(y_pred_OverSampling, return_counts=True))
print("True distribution: ", np.unique(y_true, return_counts=True))

Unique predictions: (array([0, 1]), array([150, 474]))
True distribution:  (array([0, 1], dtype=int32), array([234, 390]))
time: 0 ns (started: 2026-04-17 17:30:41 +05:30)


In [21]:
print("Report:\n",confusion_matrix(y_true, y_pred_OverSampling))

Report:
 [[135  99]
 [ 15 375]]
time: 0 ns (started: 2026-04-17 17:30:41 +05:30)


## Evalute K-Fold

In [22]:
model_fold1 = load_model("best_model_fold_1.h5")
model_fold2 = load_model("best_model_fold_2.h5")
model_fold3 = load_model("best_model_fold_3.h5")

time: 1.38 s (started: 2026-04-17 17:30:41 +05:30)


In [23]:
test_generator.reset()
y_true = test_generator.classes

time: 0 ns (started: 2026-04-17 17:30:43 +05:30)


In [24]:
test_generator.reset()
raw_preds_fold1 = model_fold1.predict(test_generator).flatten()

test_generator.reset()
raw_preds_fold2 = model_fold2.predict(test_generator).flatten()

test_generator.reset()
raw_preds_fold3 = model_fold3.predict(test_generator).flatten()

20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 219ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 223ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 237ms/step
time: 17.1 s (started: 2026-04-17 17:30:43 +05:30)


In [25]:
raw_preds_kfold = (raw_preds_fold1 + raw_preds_fold2 + raw_preds_fold3) / 3

time: 0 ns (started: 2026-04-17 17:31:00 +05:30)


In [26]:
def find_best_threshold(y_true, raw_preds, name="Model"):
    best_thresh = 0.5
    best_f1 = 0

    for t in np.arange(0.2, 0.8, 0.01):
        preds = (raw_preds > t).astype(int)
        f1 = f1_score(y_true, preds)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    print(f"{name} → Best Threshold: {best_thresh:.2f}, F1: {best_f1:.4f}")
    return best_thresh

time: 0 ns (started: 2026-04-17 17:31:00 +05:30)


In [27]:
t1 = find_best_threshold(y_true, raw_preds_fold1, "Fold 1")
t2 = find_best_threshold(y_true, raw_preds_fold2, "Fold 2")
t3 = find_best_threshold(y_true, raw_preds_fold3, "Fold 3")
t_ens = find_best_threshold(y_true, raw_preds_kfold, "Ensemble")

Fold 1 → Best Threshold: 0.80, F1: 0.9096
Fold 2 → Best Threshold: 0.74, F1: 0.9070
Fold 3 → Best Threshold: 0.76, F1: 0.8943
Ensemble → Best Threshold: 0.79, F1: 0.9075
time: 609 ms (started: 2026-04-17 17:31:00 +05:30)


In [28]:
preds_fold1 = (raw_preds_fold1 > t1).astype(int)
preds_fold2 = (raw_preds_fold2 > t2).astype(int)
preds_fold3 = (raw_preds_fold3 > t3).astype(int)
preds_kfold = (raw_preds_kfold > t_ens).astype(int)

time: 0 ns (started: 2026-04-17 17:31:00 +05:30)


In [29]:
def evaluate_model(y_true, preds, name="Model"):
    print(f"\n-------------{name}-------------")
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, preds))
    
    print("\nClassification Report:")
    print(classification_report(y_true, preds))

time: 0 ns (started: 2026-04-17 17:31:00 +05:30)


In [30]:
evaluate_model(y_true, preds_fold1, "Fold 1")
evaluate_model(y_true, preds_fold2, "Fold 2")
evaluate_model(y_true, preds_fold3, "Fold 3")
evaluate_model(y_true, preds_kfold, "Ensemble (KFold Avg)")


-------------Fold 1-------------
Confusion Matrix:
[[196  38]
 [ 33 357]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.84      0.85       234
           1       0.90      0.92      0.91       390

    accuracy                           0.89       624
   macro avg       0.88      0.88      0.88       624
weighted avg       0.89      0.89      0.89       624


-------------Fold 2-------------
Confusion Matrix:
[[189  45]
 [ 29 361]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.81      0.84       234
           1       0.89      0.93      0.91       390

    accuracy                           0.88       624
   macro avg       0.88      0.87      0.87       624
weighted avg       0.88      0.88      0.88       624


-------------Fold 3-------------
Confusion Matrix:
[[174  60]
 [ 26 364]]

Classification Report:
              precision    recall  f1-score   sup

## Evaluate K-Fold Custom

In [31]:
test_generator.reset()
y_true = test_generator.classes

time: 0 ns (started: 2026-04-17 17:31:00 +05:30)


In [32]:
model_kfoldc1 = load_model("best_model_kfoldc_1.h5")
model_kfoldc2 = load_model("best_model_kfoldc_2.h5")
model_kfoldc3 = load_model("best_model_kfoldc_3.h5")

time: 1.41 s (started: 2026-04-17 17:31:00 +05:30)


In [33]:
test_generator.reset()
raw_preds_kfoldc1 = model_kfoldc1.predict(test_generator).flatten()

test_generator.reset()
raw_preds_kfoldc2 = model_kfoldc2.predict(test_generator).flatten()

test_generator.reset()
raw_preds_kfoldc3 = model_kfoldc3.predict(test_generator).flatten()

20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 224ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 226ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 226ms/step
time: 17.1 s (started: 2026-04-17 17:31:02 +05:30)


In [34]:
raw_preds_kfoldc = (raw_preds_kfoldc1 + raw_preds_kfoldc2 + raw_preds_kfoldc3) / 3

time: 0 ns (started: 2026-04-17 17:31:19 +05:30)


In [35]:
tc1    = find_best_threshold(y_true, raw_preds_kfoldc1, "KFold Custom Fold 1")
tc2    = find_best_threshold(y_true, raw_preds_kfoldc2, "KFold Custom Fold 2")
tc3    = find_best_threshold(y_true, raw_preds_kfoldc3, "KFold Custom Fold 3")
tc_ens = find_best_threshold(y_true, raw_preds_kfoldc,  "KFold Custom Ensemble")

KFold Custom Fold 1 → Best Threshold: 0.59, F1: 0.9127
KFold Custom Fold 2 → Best Threshold: 0.80, F1: 0.8794
KFold Custom Fold 3 → Best Threshold: 0.60, F1: 0.9133
KFold Custom Ensemble → Best Threshold: 0.76, F1: 0.9128
time: 656 ms (started: 2026-04-17 17:31:19 +05:30)


In [36]:
preds_kfoldc1 = (raw_preds_kfoldc1 > tc1).astype(int)
preds_kfoldc2 = (raw_preds_kfoldc2 > tc2).astype(int)
preds_kfoldc3 = (raw_preds_kfoldc3 > tc3).astype(int)
preds_kfoldc  = (raw_preds_kfoldc  > tc_ens).astype(int)

time: 15 ms (started: 2026-04-17 17:31:20 +05:30)


In [37]:
evaluate_model(y_true, preds_kfoldc1, "KFold Custom Fold 1")
evaluate_model(y_true, preds_kfoldc2, "KFold Custom Fold 2")
evaluate_model(y_true, preds_kfoldc3, "KFold Custom Fold 3")
evaluate_model(y_true, preds_kfoldc,  "Ensemble (KFold Custom Avg)")


-------------KFold Custom Fold 1-------------
Confusion Matrix:
[[188  46]
 [ 24 366]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.80      0.84       234
           1       0.89      0.94      0.91       390

    accuracy                           0.89       624
   macro avg       0.89      0.87      0.88       624
weighted avg       0.89      0.89      0.89       624


-------------KFold Custom Fold 2-------------
Confusion Matrix:
[[136  98]
 [  7 383]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.58      0.72       234
           1       0.80      0.98      0.88       390

    accuracy                           0.83       624
   macro avg       0.87      0.78      0.80       624
weighted avg       0.85      0.83      0.82       624


-------------KFold Custom Fold 3-------------
Confusion Matrix:
[[179  55]
 [ 16 374]]

Classification Report:
          

## Comparision

In [38]:
# ── Comparison Table ─────────────────────────────────────────────────────────
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def summarise(name, y_true, preds, threshold, note=""):
    return {
        "Model"          : name,
        "Threshold"      : f"{threshold:.2f}",
        "Accuracy"       : f"{accuracy_score(y_true, preds):.4f}",
        "Precision (N)"  : f"{precision_score(y_true, preds, pos_label=0):.4f}",
        "Recall (N)"     : f"{recall_score(y_true, preds, pos_label=0):.4f}",
        "F1 (N)"         : f"{f1_score(y_true, preds, pos_label=0):.4f}",
        "Precision (P)"  : f"{precision_score(y_true, preds, pos_label=1):.4f}",
        "Recall (P)"     : f"{recall_score(y_true, preds, pos_label=1):.4f}",
        "F1 (P)"         : f"{f1_score(y_true, preds, pos_label=1):.4f}",
        "Macro F1"       : f"{f1_score(y_true, preds, average='macro'):.4f}",
        "Note"           : note,
    }

rows = [
    summarise("Class Weights",          y_true, y_pred_class_weights.flatten(), best_thresh_class_weights),
    summarise("OverSampling",           y_true, y_pred_OverSampling.flatten(),  best_thresh_OverSampling),
    summarise("KFold Fold 1",           y_true, preds_fold1,  t1),
    summarise("KFold Fold 2",           y_true, preds_fold2,  t2),
    summarise("KFold Fold 3",           y_true, preds_fold3,  t3),
    summarise("KFold Ensemble",         y_true, preds_kfold,  t_ens, note="avg of 3 folds"),
    summarise("KFold Custom Fold 1",    y_true, preds_kfoldc1, tc1),
    summarise("KFold Custom Fold 2",    y_true, preds_kfoldc2, tc2),
    summarise("KFold Custom Fold 3",    y_true, preds_kfoldc3, tc3),
    summarise("KFold Custom Ensemble",  y_true, preds_kfoldc,  tc_ens, note="avg of 3 folds"),
]

import pandas as pd
df = pd.DataFrame(rows).set_index("Model")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
print(df.to_string())

                      Threshold Accuracy Precision (N) Recall (N)  F1 (N) Precision (P) Recall (P)  F1 (P) Macro F1            Note
Model                                                                                                                              
Class Weights              0.79   0.7740        0.8496     0.4829  0.6158        0.7536     0.9487  0.8400   0.7279                
OverSampling               0.66   0.8173        0.9000     0.5769  0.7031        0.7911     0.9615  0.8681   0.7856                
KFold Fold 1               0.80   0.8862        0.8559     0.8376  0.8467        0.9038     0.9154  0.9096   0.8781                
KFold Fold 2               0.74   0.8814        0.8670     0.8077  0.8363        0.8892     0.9256  0.9070   0.8717                
KFold Fold 3               0.76   0.8622        0.8700     0.7436  0.8018        0.8585     0.9333  0.8943   0.8481                
KFold Ensemble             0.79   0.8830        0.8578     0.8248  0.8410   